> **Historical study — preserved evidence.** This notebook records the original P3 exploration and outputs. Workstation paths and transient runtime metadata were redacted for publication. The maintained, typed implementation lives in `src/off_quality`; the historical notebook is retained to demonstrate the breadth and evolution of the work.

In [1]:
import matplotlib.figure as fig
from matplotlib.patches import Polygon
import pandas as pa
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
off = pa.read_csv('../fr.openfoodfacts.org.products/fr.openfoodfacts.org.products.csv', sep='\t')

<LOCAL_PATH>
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
off_fr = off.where(off.countries.str.contains('fr', flags=re.IGNORECASE, regex=True)).dropna(subset=['countries']).copy()

In [4]:
def mean_col_missing(x):
    return x.isnull().mean()

def select_threshold(data, seuil):
    data = data[data.apply(mean_col_missing).where(data.apply(mean_col_missing) < seuil).dropna().keys()].copy()
    return data
def select_drop(x, i_rows):
    y = x.drop(i_rows).copy()
    return y

def list_duplicated(x):
   return x[x.duplicated()]

def describe_columns(x):
    return x.describe()

In [5]:
off_fr_cp = off_fr.copy()

off_fr_cp

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,product_name,generic_name,quantity,...,ph_100g,fruits-vegetables-nuts_100g,collagen-meat-protein-ratio_100g,cocoa_100g,chlorophyl_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,glycemic-index_100g,water-hardness_100g
0,3087,http://world-fr.openfoodfacts.org/produit/0000...,openfoodfacts-contributors,1474103866,2016-09-17T09:17:46Z,1474103893,2016-09-17T09:18:13Z,Farine de blé noir,NaN,1kg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,24600,http://world-fr.openfoodfacts.org/produit/0000...,date-limite-app,1434530704,2015-06-17T08:45:04Z,1434535914,2015-06-17T10:11:54Z,Filet de bœuf,NaN,2.46 kg,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
48,27205,http://world-fr.openfoodfacts.org/produit/0000...,tacinte,1458238630,2016-03-17T18:17:10Z,1458238638,2016-03-17T18:17:18Z,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,36252,http://world-fr.openfoodfacts.org/produit/0000...,tacinte,1422221701,2015-01-25T21:35:01Z,1489055667,2017-03-09T10:34:27Z,Lion Peanut x2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,22.0,22.0,NaN,NaN
136,39259,http://world-fr.openfoodfacts.org/produit/0000...,tacinte,1422221773,2015-01-25T21:36:13Z,1473538082,2016-09-10T20:08:02Z,Twix x2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
320761,9906410000009,http://world-fr.openfoodfacts.org/produit/9906...,agamitsudo,1373480408,2013-07-10T18:20:08Z,1451851215,2016-01-03T20:00:15Z,Roussette du Bugey (2011),Vins blanc du Bugey,750 ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
320763,99111250,http://world-fr.openfoodfacts.org/produit/9911...,balooval,1367163039,2013-04-28T15:30:39Z,1371690556,2013-06-20T01:09:16Z,Thé vert Earl grey,thé bio équitable,50 g,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.0,NaN,NaN
320764,9918,http://world-fr.openfoodfacts.org/produit/9918...,woshilapin,1430167954,2015-04-27T20:52:34Z,1430167992,2015-04-27T20:53:12Z,"Cheese cake thé vert, yuzu",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
320765,9935010000003,http://world-fr.openfoodfacts.org/produit/9935...,sebleouf,1446293229,2015-10-31T12:07:09Z,1446376839,2015-11-01T11:20:39Z,Rillette d'oie,NaN,180 g,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
off_fr_cp.dtypes.value_counts()

float64    106
object      56
dtype: int64

In [7]:
for x in off_fr_cp.columns:
    if off_fr_cp[x].dtypes == object:
        print(x)

code
url
creator
created_t
created_datetime
last_modified_t
last_modified_datetime
product_name
generic_name
quantity
packaging
packaging_tags
brands
brands_tags
categories
categories_tags
categories_fr
origins
origins_tags
manufacturing_places
manufacturing_places_tags
labels
labels_tags
labels_fr
emb_codes
emb_codes_tags
first_packaging_code_geo
cities
cities_tags
purchase_places
stores
countries
countries_tags
countries_fr
ingredients_text
allergens
allergens_fr
traces
traces_tags
traces_fr
serving_size
additives
additives_tags
additives_fr
ingredients_from_palm_oil_tags
ingredients_that_may_be_from_palm_oil_tags
nutrition_grade_fr
pnns_groups_1
pnns_groups_2
states
states_tags
states_fr
main_category
main_category_fr
image_url
image_small_url


In [32]:
index = off_fr_cp.columns.where(off_fr_cp.dtypes == object).dropna()
off_qual = off_fr_cp[index].copy()
off_qual_clean = select_threshold(off_qual,0.50)
off_qual_des = off_qual.describe()
off_qual_des.loc['freq']

code                                              2
url                                               1
creator                                       29195
created_t                                         6
created_datetime                                  6
last_modified_t                                  15
last_modified_datetime                           15
product_name                                     62
generic_name                                    201
quantity                                       3297
packaging                                      2104
packaging_tags                                 3873
brands                                         2838
brands_tags                                    2977
categories                                      291
categories_tags                                 580
categories_fr                                   580
origins                                        5060
origins_tags                                   5188
manufacturin

In [34]:
off_qual.additives_tags.dropna()

106                                        en:e322
184                       en:e1400,en:e322,en:e503
189                en:e330,en:e171,en:e211,en:e131
226                               en:e150d,en:e338
240                        en:e330,en:e296,en:e331
                            ...                   
320626                    en:e330,en:e300,en:e150d
320630                    en:e260,en:e415,en:e14xx
320652                                     en:e500
320681            en:e420,en:e955,en:e950,en:e470b
320702    en:e420,en:e955,en:e950,en:e470b,en:e330
Name: additives_tags, Length: 30473, dtype: object

In [9]:
off_qual.generic_name.unique()

array([nan, 'Biscuits sablés déclassés fourrage au cacao',
       'Bonbons acidulés Raisin Fraise', ..., 'semoule de manioc',
       'Vins blanc du Bugey', 'thé bio équitable'], dtype=object)

In [10]:
off_qual.main_category_fr.isna().value_counts()

False    61981
True     36487
Name: main_category_fr, dtype: int64

In [11]:
pa.DataFrame(off_qual.generic_name).join(pa.DataFrame(off_qual.main_category_fr)).isna().value_counts()

generic_name  main_category_fr
False         False               37957
True          True                36125
              False               24024
False         True                  362
dtype: int64

In [12]:
off_qual.main_category_fr.unique()

array([nan, 'Filet-de-boeuf', 'Aliments et boissons à base de végétaux',
       ..., 'en:Cremes-vegetales-a-base-de-coco-pour-cuisiner',
       'en:Malt-vinegar', 'Attieke'], dtype=object)

In [13]:
def bag_of_word(x):
    z = x.split()
    return z

In [14]:
off_qual_categ = off_qual.dropna(subset=['main_category_fr']).copy()
off_qual_categ.main_category_fr.apply(bag_of_word)

46                                       [Filet-de-boeuf]
182       [Aliments, et, boissons, à, base, de, végétaux]
183                                           [Root-bier]
184                                              [Sablés]
187                                             [Bonbons]
                               ...                       
320755                               [Pâtes, à, tartiner]
320758                             [Produits, d'élevages]
320761                                         [Boissons]
320763                                      [Thés, verts]
320765                     [Produits, à, tartiner, salés]
Name: main_category_fr, Length: 61981, dtype: object

In [146]:
def my_tokenizer(text):
    # create a space between special characters
    text=re.sub("(^[a-zA-Z0-9À-ÿ]+$)","",text)
    # text=re.sub("(\\W+)","",text)

    # split based on whitespace
    mot = re.split("\\s+",text)
    # mot = text
    return mot #.replace("-","_")

data = off_qual_categ.main_category_fr
# Create the bag of words feature matrix
# count = CountVectorizer(stop_words={'french'}, ngram_range=(1, 1), token_pattern=r'^[a-zA-Z0-9À-ÿ]+$')
# count = CountVectorizer(token_pattern=r'\b[^\d\W]+\b/g')
# count = CountVectorizer()
count = CountVectorizer(stop_words={'french'},tokenizer=my_tokenizer)


bag_of_words = count.fit_transform(data)

# Show feature matrix
bag_of_words.toarray()

# Get feature names
feature_names = count.get_feature_names_out()

# View feature names
feature_names
# Create data frame
sac_de_mot = pa.DataFrame(bag_of_words.toarray(), columns=feature_names)
sac_de_mot

<LOCAL_PATH>
  warnings.warn(


,,100%,2000,a-code-1,abats-surgeles,accras-de-morue,acras-de-morue,additifs,aide-a-la-perte-de-poids,aide-culinaire,...,yakitori-de-poulet,yaourt-au-lait-d-amande,yaourt-de-soja-a-la-vanille,yaourt-maigre,yaourts,yaourts-a-la-grecque-2-parfums-sur-lit-de-fruits-jaunes,yaourts-nature-alleges,yaurts-natures,zh:sauce-de-soja,à
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61976,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
61977,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
61978,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
61979,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
sac_de_mot.sum()

                                                           35897
100%                                                         574
2000                                                          17
a-code-1                                                       1
abats-surgeles                                                 1
                                                           ...  
yaourts-a-la-grecque-2-parfums-sur-lit-de-fruits-jaunes        1
yaourts-nature-alleges                                         1
yaurts-natures                                                 1
zh:sauce-de-soja                                               1
à                                                           5649
Length: 1363, dtype: int64

In [149]:
import nltk
nltk.download('punkt')
tokenizer = nltk.RegexpTokenizer(r'\w+')
tokenizer.tokenize("Bonjour, je suis un texte d'exemple pour le cours d'Openclassrooms. Soyez attentifs à ce cours !")

[nltk_data] Downloading package punkt to
[nltk_data]     <LOCAL_PATH>\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['Bonjour',
 'je',
 'suis',
 'un',
 'texte',
 'd',
 'exemple',
 'pour',
 'le',
 'cours',
 'd',
 'Openclassrooms',
 'Soyez',
 'attentifs',
 'à',
 'ce',
 'cours']

In [118]:
def my_tokenizer(text):
    # text=re.sub("e\d{3}"," ",text)
    mot = re.split("e\d{3}",text)
    # mot = text
    print(mot)
    return mot #.replace("-","_")

addit = off_qual.additives_tags.dropna()

In [ ]:
addit

In [120]:
# Create the bag of words feature matrix
# count = CountVectorizer(stop_words={'french'}, ngram_range=(1, 1), token_pattern=r'^[a-zA-Z0-9À-ÿ]+$')
count = CountVectorizer(stop_words={'french'}, ngram_range=(1, 1), token_pattern=r'e\d{3}')
# count = CountVectorizer()
# count = CountVectorizer(tokenizer=my_tokenizer)
b_o_w = count.fit_transform(addit)

In [94]:
b_o_w.toarray()

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [123]:
feature_names = count.get_feature_names_out()
feature_names

array(['e100', 'e101', 'e102', 'e104', 'e110', 'e120', 'e122', 'e123',
       'e124', 'e127', 'e129', 'e130', 'e131', 'e132', 'e133', 'e140',
       'e141', 'e142', 'e144', 'e145', 'e150', 'e151', 'e152', 'e153',
       'e155', 'e160', 'e161', 'e162', 'e163', 'e166', 'e170', 'e171',
       'e172', 'e173', 'e174', 'e175', 'e180', 'e200', 'e201', 'e202',
       'e203', 'e210', 'e211', 'e212', 'e218', 'e220', 'e221', 'e222',
       'e223', 'e224', 'e225', 'e228', 'e230', 'e231', 'e232', 'e233',
       'e234', 'e235', 'e236', 'e239', 'e242', 'e249', 'e250', 'e251',
       'e252', 'e260', 'e261', 'e262', 'e263', 'e270', 'e280', 'e281',
       'e282', 'e285', 'e290', 'e296', 'e297', 'e300', 'e301', 'e302',
       'e304', 'e306', 'e307', 'e309', 'e310', 'e315', 'e316', 'e319',
       'e320', 'e321', 'e322', 'e325', 'e326', 'e327', 'e329', 'e330',
       'e331', 'e332', 'e333', 'e334', 'e335', 'e336', 'e338', 'e339',
       'e340', 'e341', 'e343', 'e345', 'e350', 'e352', 'e363', 'e375',
      

In [128]:
sac_de_mot = pa.DataFrame(b_o_w.toarray(), columns=feature_names)
sac_de_mot.sum()

e100    769
e101    723
e102     81
e104     15
e110    355
       ... 
e965    230
e966      1
e967     70
e968     20
e999      3
Length: 256, dtype: int64

array(['code', 'url', 'creator', 'created_t', 'created_datetime',
       'last_modified_t', 'last_modified_datetime', 'product_name',
       'generic_name', 'quantity', 'packaging', 'packaging_tags',
       'brands', 'brands_tags', 'categories', 'categories_tags',
       'categories_fr', 'origins', 'origins_tags', 'manufacturing_places',
       'manufacturing_places_tags', 'labels', 'labels_tags', 'labels_fr',
       'emb_codes', 'emb_codes_tags', 'first_packaging_code_geo',
       'cities', 'cities_tags', 'purchase_places', 'stores', 'countries',
       'countries_tags', 'countries_fr', 'ingredients_text', 'allergens',
       'allergens_fr', 'traces', 'traces_tags', 'traces_fr',
       'serving_size', 'no_nutriments', 'additives_n', 'additives',
       'additives_tags', 'additives_fr', 'ingredients_from_palm_oil_n',
       'ingredients_from_palm_oil', 'ingredients_from_palm_oil_tags',
       'ingredients_that_may_be_from_palm_oil_n',
       'ingredients_that_may_be_from_palm_oil',
   